# Replicate Iceberg `migrate` procedure behavior in Glue Data Catalog

In [ ]:
%%configure -f
{   
    "numExecutors": 1, 
     "executorMemory": "14G",
     "executorCores": 4,
    "conf":{
        "spark.dynamicAllocation.enabled": "false",
         "spark.sql.extensions":"org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions",
         "spark.sql.catalog.spark_catalog":"org.apache.iceberg.spark.SparkSessionCatalog",
         "spark.sql.catalog.spark_catalog.type":"glue",
         "spark.hadoop.hive.metastore.client.factory.class":"com.amazonaws.glue.catalog.metastore.AWSGlueDataCatalogHiveClientFactory"
    }
}

Setup variables

In [ ]:
DATABASE_NAME="mydb2"
S3_BUCKET_NAME="my_datalake"
SOURCE_TABLE_NAME="sales_transactions_3"
TEMP_TABLE_NAME="sales_transactions_iceberg_snapshot_temp"
BACKUP_TABLE_NAME="sales_transactions_3_backup"
AWS_REGION="us-east-1"

## 1. Create the source table

In [ ]:
# Create database (if not already created)
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
LOCATION 's3://{S3_BUCKET_NAME}/{DATABASE_NAME}/'
""")

In [ ]:
# Create table with day partition
spark.sql(f"""
CREATE EXTERNAL TABLE IF NOT EXISTS {DATABASE_NAME}.{SOURCE_TABLE_NAME} (
    transaction_id STRING,
    total_amount DECIMAL(10,2),
    transaction_time TIMESTAMP,
    day STRING
)
PARTITIONED BY (day)
STORED AS PARQUET
LOCATION 's3://{S3_BUCKET_NAME}/{DATABASE_NAME}/{SOURCE_TABLE_NAME}/'
""")

## 2. Populate the source table

Here we create 10 daily partitions. Each partition with 100 files.

In [ ]:
from pyspark.sql.types import *
import random
from decimal import Decimal
from datetime import datetime, timedelta

spark.conf.set("hive.exec.dynamic.partition.mode", "nonstrict")

# Function to generate random transactions for a specific day
def generate_transactions_for_day(date_str, num_transactions):
    data = []
    
    # Convert date_str (YYYYMMDD) to datetime
    year = int(date_str[:4])
    month = int(date_str[4:6])
    day = int(date_str[6:])
    base_date = datetime(year, month, day)
    
    for _ in range(num_transactions):
        transaction_id = f"TXN_AR_{random.randint(10000000, 99999999)}"
        # Create Decimal directly from random.uniform without using round function
        total_amount = Decimal(str(random.uniform(10.00, 1000.00))).quantize(Decimal('.01'))
        
        # Random time during the day
        hour = random.randint(0, 23)
        minute = random.randint(0, 59)
        second = random.randint(0, 59)
        
        transaction_time = datetime(year, month, day, hour, minute, second)
        
        data.append((transaction_id, total_amount, transaction_time, date_str))
    
    return data

# Schema for the DataFrame
schema = StructType([
    StructField("transaction_id", StringType(), False),
    StructField("total_amount", DecimalType(10,2), False),
    StructField("transaction_time", TimestampType(), False),
    StructField("day", StringType(), False)
])

# Base date to start from (current date)
base_date = datetime(2025, 9, 1)

# Write data for 10 days
for i in range(10):
    current_date = base_date + timedelta(days=i)
    date_str = current_date.strftime("%Y%m%d")
    
    # Generate 1000 transactions for the day
    transactions = generate_transactions_for_day(date_str, 1000)
    
    # Create DataFrame
    df = spark.createDataFrame(transactions, schema)
    
    # Write to table
    df.repartition(100).write.mode("append").insertInto(f"{DATABASE_NAME}.{SOURCE_TABLE_NAME}")
    
    print(f"Loaded {len(transactions)} transactions for day {date_str}")

print("Data generation completed!")

## 3. Create a temporary Iceberg table using the snapshot procedure

In [ ]:
spark.sql(f"""
CALL system.snapshot(
    source_table => '{DATABASE_NAME}.{SOURCE_TABLE_NAME}',
    table => '{DATABASE_NAME}.{TEMP_TABLE_NAME}',
    location => 's3://{S3_BUCKET_NAME}/{DATABASE_NAME}/{SOURCE_TABLE_NAME}/'
)
""")

## 4. Backup the source table

In [ ]:
import boto3
import os

def get_glue_client():
    return boto3.client('glue', region_name=AWS_REGION)

def copy_table_metadata(source_database, source_table, target_database, target_table):
    glue_client = get_glue_client()
    try:
        # Get source table metadata
        source = glue_client.get_table(DatabaseName=source_database, Name=source_table)
        table_input = source['Table']
        
        # Prepare new table input with required fields
        new_table_input = {
            'Name': target_table,
            'StorageDescriptor': table_input['StorageDescriptor'],
            'PartitionKeys': table_input['PartitionKeys'],
            'TableType': table_input['TableType'],
            'Parameters': table_input['Parameters']
        }
        
        # Handle optional fields
        optional_fields = ['Owner', 'Description', 'Retention', 'ViewOriginalText', 
                         'ViewExpandedText', 'TargetTable']
        for field in optional_fields:
            if field in table_input:
                new_table_input[field] = table_input[field]
        # Create the backup table
        glue_client.create_table(
            DatabaseName=target_database,
            TableInput=new_table_input
        )
        
        print(f"Table metadata successfully copied from {source_database}.{source_table} to {target_database}.{target_table}")
        return True
    except Exception as e:
        print(f"Error copying table metadata: {str(e)}")
        return False

def copy_partition_metadata(source_database, source_table, target_database, target_table):
    glue_client = get_glue_client()
    try:
        # Set up pagination for getting partitions
        paginator = glue_client.get_paginator('get_partitions')
        partition_iterator = paginator.paginate(
            DatabaseName=source_database,
            TableName=source_table
        )
        # Copy partitions page by page
        partition_count = 0
        for page in partition_iterator:
            for partition in page['Partitions']:
                partition_input = {
                    'Values': partition['Values'],
                    'StorageDescriptor': partition['StorageDescriptor']
                }
                
                glue_client.create_partition(
                    DatabaseName=target_database,
                    TableName=target_table,
                    PartitionInput=partition_input
                )
                partition_count += 1
        
        print(f"Successfully copied {partition_count} partitions from {source_database}.{source_table} to {target_database}.{target_table}")
        return True
    except Exception as e:
        print(f"Error copying partition metadata: {str(e)}")
        return False

def backup_table(source_database, source_table, target_database, target_table):
    print(f"Starting backup of {source_database}.{source_table} to {target_database}.{target_table}")
    
    if copy_table_metadata(source_database, source_table, target_database, target_table):
        if copy_partition_metadata(source_database, source_table, target_database, target_table):
            print(f"Backup of {source_database}.{source_table} to {target_database}.{target_table} completed successfully")
            return True
        else:
            print(f"Backup failed: Unable to copy partition metadata")
            return False
    else:
        print(f"Backup failed: Unable to copy table metadata")
        return False

In [ ]:
# Execute the backup
backup_table(DATABASE_NAME, SOURCE_TABLE_NAME, DATABASE_NAME, BACKUP_TABLE_NAME)

In [ ]:
#Verify the backup table is ok
spark.sql(f"""
select * from {DATABASE_NAME}.{BACKUP_TABLE_NAME}
""").show(truncate=False)

## 5. Drop the source table

In [ ]:
spark.sql(f"""
DROP TABLE {DATABASE_NAME}.{SOURCE_TABLE_NAME}
""")

## 6. Get Iceberg table JSON metadata file location & Register new Iceberg table

In [ ]:
import boto3

s3_client = boto3.client('s3', region_name=AWS_REGION)
# Set prefix
prefix = f"{DATABASE_NAME}/{SOURCE_TABLE_NAME}/metadata/"

# List objects with .json extension
response = s3_client.list_objects_v2(
    Bucket=S3_BUCKET_NAME,
    Prefix=prefix
)

# Filter for JSON files and read their content
for obj in response.get('Contents', []):
    if obj['Key'].endswith('.json'):
        metadata_file=f"s3://{S3_BUCKET_NAME}/{obj['Key']}"
        print(f"File: {metadata_file}")
        break

In [ ]:
# Register the Iceberg table with the original table name
spark.sql(f"""
    CALL system.register_table(
        table => '{DATABASE_NAME}.{SOURCE_TABLE_NAME}', 
        metadata_file => '{metadata_file}'
    )
""").show(truncate=False)

In [ ]:
# Validate the new table
spark.sql(f"""
    SELECT * FROM {DATABASE_NAME}.{SOURCE_TABLE_NAME}
""").show(truncate=False)

## 7. Enable full operations on Iceberg table

In [ ]:
# Allow operations that can physically delete data files
spark.sql(f"""
    ALTER TABLE {DATABASE_NAME}.{SOURCE_TABLE_NAME}
    SET TBLPROPERTIES ('gc.enabled' = 'true')
""")

## 8. Drop the temp Iceberg table

In [ ]:
spark.sql(f"""
DROP TABLE {DATABASE_NAME}.{TEMP_TABLE_NAME}
""")